# Logistic regression baseline

Start with one unweighted L2 logistic regression (fixed C=1) and a training-prior
probability reference. Estimate purchase probabilities before designing campaign
actions. No hyperparameter search, class weighting, threshold tuning or cost policy.


In [1]:
from pathlib import Path
from importlib import import_module, reload

import matplotlib.pyplot as plt
import pandas as pd

for name in ("preprocessing", "metrics", "baseline"):
    module = reload(import_module(f"online_retail.modeling.{name}"))
run_baseline = module.run_baseline
plot_calibration = module.plot_calibration

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/processed"
OUTPUT = ROOT / "artifacts/logistic_baseline"


## Data and class balance

Rows are customer-week snapshots. Each row uses past-only features and a fully
observed 50-day outcome. Class 1 = purchase in days 0–49; class 0 = no
purchase during that interval. Keep the observed proportions; do not rebalance this baseline.


In [2]:
train = pd.read_parquet(DATA / "train.parquet")
counts = train["RepurchaseWithin50Days"].value_counts().reindex([0, 1], fill_value=0)
pd.DataFrame({"snapshots": counts, "share_pct": counts / len(train) * 100}).round(2)


,snapshots,share_pct
RepurchaseWithin50Days,,
0,55824,60.47
1,36495,39.53


## Repeated snapshots and evaluation

Many weekly rows belong to the same customer. Adjacent feature histories and 50-day
outcomes overlap, so row count overstates independent information. Longer-observed
customers contribute more rows. Repetition does not automatically imply overfitting.

Keep weekly scoring for now. Use expanding date-based folds with training labels
mature before validation starts; never randomly split rows. L2 regularization limits
coefficient growth. Compare training and later validation losses, recognizing that
a gap can reflect time drift as well as overfitting. Do not use independent-row
confidence intervals. A later sensitivity check can thin training cutoffs to every
fourth week while leaving evaluation dates unchanged; no cadence change is made now.


In [3]:
pd.Series({
    "training_snapshots": len(train),
    "distinct_customers": train["Customer ID"].nunique(),
    "scoring_weeks": train["cutoff"].nunique(),
    "median_snapshots_per_customer": train.groupby("Customer ID").size().median(),
})


training_snapshots               92319.0
distinct_customers                3687.0
scoring_weeks                       33.0
median_snapshots_per_customer       31.0
dtype: float64

## Fit the baseline

Use 22 original behavioral/calendar predictors, excluding derived RFM scores/segment.
Imputation, predefined log1p count/spend transforms and scaling fit inside each fold.
The three internal validation windows cover July, August and September–October 2010;
training requires cutoff + 50 days <= validation start. Then fit initial training
and assess the separate December–March validation period. Final refit/test stay unused.


In [4]:
baseline = run_baseline(DATA, OUTPUT)
baseline["cv_metrics"].round(4)


,fold,model,train_log_loss,log_loss,weekly_log_loss,brier_score,accuracy,balanced_accuracy,macro_f1
0,1,logistic_l2,0.5989,0.5354,0.5354,0.1777,0.7387,0.6528,0.6617
1,1,prior,0.6832,0.6577,0.6577,0.2324,0.6604,0.5000,0.3977
2,2,logistic_l2,0.5862,0.5964,0.5958,0.2000,0.7159,0.6370,0.6353
3,2,prior,0.6783,0.6634,0.6633,0.2352,0.6290,0.5000,0.3861
4,3,logistic_l2,0.5694,0.7196,0.7186,0.2443,0.6565,0.6131,0.5870
5,3,prior,0.6705,0.6878,0.6877,0.2473,0.5658,0.5000,0.3614


In [5]:
# Probability metrics are primary; argmax classification metrics are diagnostic only.
baseline["validation_metrics"].round(4)


,model,log_loss,weekly_log_loss,brier_score,accuracy,balanced_accuracy,macro_f1
0,prior,0.6001,0.5999,0.2041,0.7717,0.5000,0.4356
1,logistic_l2,0.6769,0.6783,0.2294,0.6682,0.7005,0.6301


## Probability calibration

Points close to the diagonal indicate agreement between predicted probabilities
and observed class frequencies. This is a diagnostic using validation data, not a
fitted recalibration. Weekly observations are correlated. Compare log loss and Brier
with the prior reference before adding a more complex model.


In [6]:
plot_calibration(baseline["validation_predictions"])
plt.show()


/tmp/ipykernel_373657/3995842565.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The pipeline, CV/validation metrics and validation probabilities are saved in
`artifacts/logistic_baseline/`. Review this baseline first. Campaign costs,
thresholds, calibration fitting, Optuna and additional models are deferred.


**Initial finding.** On external validation, the binary logistic baseline has log loss
0.677 and Brier score 0.229, versus 0.600 and 0.204 for the prior reference (lower
is better). Balanced accuracy improves to 0.701 versus 0.500, but probability quality
does not improve. The validation non-purchase share is 77.2%, compared with 60.5%
in training, so outcome prevalence changes substantially over time.

Simplifying the target has not by itself resolved the baseline's limitations.
Inspect calibration and temporal drift before adding complexity. These results do
not establish that weekly snapshots cause overfitting. The final test is unevaluated.
